# Judge validation: human vs. LLM-judge agreement

In [ ]:
import json
import random
from pathlib import Path

import pandas as pd

EVAL_DIR = Path.cwd() / ".." / "data" / "eval" / "end_to_end"

with open(EVAL_DIR / "generation_answers.json", encoding="utf-8") as f:
    answers = json.load(f)
with open(EVAL_DIR / "generation_judged.json", encoding="utf-8") as f:
    judged = json.load(f)

answers_by_key = {(a["id"], a["top_n"]): a for a in answers}
judged_by_key = {(j["id"], j["top_n"]): j for j in judged}

print("answers:", len(answers), "| judged:", len(judged))

## Display a sample

In [ ]:
import html as _html
from IPython.display import HTML, display

TOP_N = 10                  # 5 or 10
TYPE = "multi_source"     # "single_source" or "multi_source"

GOLD_DIR = EVAL_DIR.parent / "test_jsonfiles" / "golden"
gold_by_id = {}
for name in ("single_source_goldset.json", "multi_source_goldset.json"):
    with open(GOLD_DIR / name, encoding="utf-8") as f:
        for g in json.load(f):
            gold_by_id[g["id"]] = g["source_chunk_ids"]

def render_answer(text):
    return "<div style='white-space:pre-wrap'>" + _html.escape(text) + "</div>"

def render_context(sources, gold_ids):
    blocks = []
    for i, s in enumerate(sources, start=1):
        gold = " (gold)" if s["chunk_id"] in gold_ids else ""
        blocks.append(
            "<div style='margin-bottom:14px'>"
            "<div style='font-weight:600'>[" + str(i) + "] " + _html.escape(s["title"])
            + " &mdash; " + _html.escape(s["chunk_id"]) + gold + "</div>"
            "<div style='white-space:pre-wrap'>" + _html.escape(s["page_content"]) + "</div>"
            "</div>"
        )
    return "".join(blocks)

pool = [a for a in answers if a["top_n"] == TOP_N and a["type"] == TYPE]
print(len(pool), "items match top_n=" + str(TOP_N), "type=" + TYPE)
item = random.choice(pool)

gold_ids = gold_by_id.get(item["id"], [])
retrieved_ids = {s["chunk_id"] for s in item["sources"]}
gold_line = ", ".join(
    cid + ("" if cid in retrieved_ids else " [not retrieved]") for cid in gold_ids
)

meta = ("<div style='color:#888;font-size:12px'>id=" + _html.escape(item["id"])
        + " · type=" + item["type"] + " · top_n=" + str(item["top_n"]) + "</div>")
question = "<h3 style='margin:4px 0 4px'>" + _html.escape(item["question"]) + "</h3>"
gold = "<div style='font-size:13px;margin-bottom:12px'><b>Gold chunks:</b> " + _html.escape(gold_line) + "</div>"
columns = (
    "<div style='display:flex;gap:16px;align-items:flex-start'>"
    "<div style='flex:1;font-size:13px;line-height:1.5'>"
    "<div style='font-weight:700;margin-bottom:8px'>CONTEXT</div>"
    + render_context(item["sources"], gold_ids) + "</div>"
    "<div style='flex:1;font-size:13px;line-height:1.5'>"
    "<div style='font-weight:700;margin-bottom:8px'>ANSWER</div>"
    + render_answer(item["answer"]) + "</div>"
    "</div>"
)
display(HTML(meta + question + gold + columns))

## My Judge

In [ ]:
my_scores = [
    {"id": "ss_15", "type": "single_source", "top_n": 5, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},
    {"id": "ss_33", "type": "single_source", "top_n": 5, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},
    {"id": "ss_49", "type": "single_source", "top_n": 5, "faithfulness": 1, "completeness": 2, "answer_relevance": 2, "final_score": 5},
    {"id": "ss_2",  "type": "single_source", "top_n": 5, "faithfulness": 1, "completeness": 2, "answer_relevance": 2, "final_score": 5},
    {"id": "ss_36", "type": "single_source", "top_n": 5, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},

    {"id": "ms_46", "type": "multi_source", "top_n": 5, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},
    {"id": "ms_42", "type": "multi_source", "top_n": 5, "faithfulness": 1, "completeness": 2, "answer_relevance": 2, "final_score": 5},
    {"id": "ms_45", "type": "multi_source", "top_n": 5, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},
    {"id": "ms_35", "type": "multi_source", "top_n": 5, "faithfulness": 1, "completeness": 2, "answer_relevance": 2, "final_score": 5},
    {"id": "ms_27", "type": "multi_source", "top_n": 5, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},

    {"id": "ss_29", "type": "single_source", "top_n": 10, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},
    {"id": "ss_38", "type": "single_source", "top_n": 10, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},
    {"id": "ss_18", "type": "single_source", "top_n": 10, "faithfulness": 1, "completeness": 2, "answer_relevance": 2, "final_score": 5},
    {"id": "ss_52", "type": "single_source", "top_n": 10, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},
    {"id": "ss_56", "type": "single_source", "top_n": 10, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},

    {"id": "ms_1",  "type": "multi_source", "top_n": 10, "faithfulness": 2, "completeness": 1, "answer_relevance": 2, "final_score": 5},
    {"id": "ms_11", "type": "multi_source", "top_n": 10, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},
    {"id": "ms_45", "type": "multi_source", "top_n": 10, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},
    {"id": "ms_22", "type": "multi_source", "top_n": 10, "faithfulness": 2, "completeness": 2, "answer_relevance": 2, "final_score": 6},
    {"id": "ms_18", "type": "multi_source", "top_n": 10, "faithfulness": 1, "completeness": 2, "answer_relevance": 2, "final_score": 5},
]


### Reasons for score reduction

Only for items where a score is below 2.

- **ss_49** (faithfulness 1): Die Antwort enthält eigene erklärende Definitionen zu den richtigen Punkten, die teils nicht im Kontext stehen.
- **ss_2** (faithfulness 1): Gängige Bilbiotheken und Python zu kennen, sowie die Fähigkeit Modelle zu verstehen und zu Hinterfragen. steht so nicht im Kontext. Inhalt wurde Interpretiert.
- **ms_42** (faithfulness 1): Anreicherung der Antwort mit Wissen aus der sklearn-Implementierung von "random".
- **ms_35** (faithfulness 1): "Speicherverbrauch" steht bei Ressourcenbedarf nicht im Label und entsprechend ungedeckter Zusatz.
- **ss_18** (faithfulness 1): Estimater= steht so nicht im Kontext. Anreichung mit externem Wissen.
- **ms_1** (completeness 1): Das Venn-Diagram aus ML_1_einleitung_page_10 (künstliche Intelligenz und machine learning) liegt zwar im Kontext, wird aber in der Antwort nicht verwendet oder referenziert. Die Frage hätte vollständiger beantwortet werden können.
- **ms_18** (faithfulness 1): Die Erhöhung des Rechenaufwands und das Risiko des Overfittings steht nicht im Kontext. Generalisierung über den Text hinaus und die Aussage ist nicht immer korrekt und allgemeingültig.

## Agreement rate

In [ ]:

my_judges = {(m["id"], m["top_n"]): m for m in my_scores}

final_result = []

for j in judged_by_key.values():
    key = (j["id"], j["top_n"])

    if key not in my_judges:
        continue

    final_result.append({
        "id": j["id"],
        "top_n": j["top_n"],

        "llm_faithfulness": j["faithfulness"],
        "human_faithfulness": my_judges[key]["faithfulness"],

        "llm_completeness": j["completeness"],
        "human_completeness": my_judges[key]["completeness"],

        "llm_relevance": j["answer_relevance"],
        "human_relevance": my_judges[key]["answer_relevance"],
    })


df = pd.DataFrame(final_result)

def agreement_rate(human, llm):
    return (human == llm).mean()

llm_all = pd.concat([
    df["llm_faithfulness"],
    df["llm_completeness"],
    df["llm_relevance"]
])

my_all = pd.concat([
    df["human_faithfulness"],
    df["human_completeness"],
    df["human_relevance"]
])

overall_agreement = (llm_all.values == my_all.values).mean()

df.to_csv(EVAL_DIR / "human_vs_llm_judges_items.csv", index=False)
display(df)

## Agreement summary

In [ ]:
def agreement_rate(human, llm):
    return (human == llm).mean()

llm_all = pd.concat([
    df["llm_faithfulness"],
    df["llm_completeness"],
    df["llm_relevance"]
])

my_all = pd.concat([
    df["human_faithfulness"],
    df["human_completeness"],
    df["human_relevance"]
])

overall_agreement = (llm_all.values == my_all.values).mean()

n = len(df) 

summary = pd.DataFrame([
    {"dimension": "Faithfulness", "agreement": agreement_rate(df["llm_faithfulness"], df["human_faithfulness"]), "n": n},
    {"dimension": "Completeness", "agreement": agreement_rate(df["llm_completeness"], df["human_completeness"]), "n": n},
    {"dimension": "Answer relevance", "agreement": agreement_rate(df["llm_relevance"], df["human_relevance"]), "n": n},
    {"dimension": "Overall", "agreement": overall_agreement, "n": 3 * n},
])
summary["agreement"] = (summary["agreement"] * 100).round(2).astype(str) + " %"

summary.to_csv(EVAL_DIR / "human_vs_llm_judges_agreement.csv", index=False)
display(summary.style.hide(axis="index"))

# Parsing judge validation


In [ ]:
import json
import random
from pathlib import Path

import pandas as pd

VERDICTS_DIR = Path.cwd() / ".." / "data" / "eval" / "parsing"

with open(VERDICTS_DIR / "parsing_judge_verdicts.json", encoding="utf-8") as f:
    verdicts = json.load(f)

verdicts_by_id = {v["id"]: v for v in verdicts}

print("verdicts:", len(verdicts))

## Display a sample

In [ ]:
import html as _html
from IPython.display import HTML, display

SAMPLE_SIZE = 20
SEED = 42
missing = [v for v in verdicts if v["verdict"] == "missing"]
covered = [v for v in verdicts if v["verdict"] == "covered"]
rng = random.Random(SEED)
sample = sorted(missing + rng.sample(covered, SAMPLE_SIZE - len(missing)), key=lambda v: v["id"])

blocks = []
for v in sample:
    meta = ("<div style='color:#888;font-size:12px'>id=" + _html.escape(str(v["id"]))
            + " · " + v["vorlesung"] + " · " + v["slide_id"] + " · typ=" + v["typ"] + "</div>")
    element = (
        "<div style='flex:1;font-size:13px;line-height:1.5'>"
        "<div style='font-weight:700;margin-bottom:8px'>ELEMENT (golden)</div>"
        "<div style='white-space:pre-wrap'>" + _html.escape(v["element"]) + "</div></div>"
    )
    parse = (
        "<div style='flex:1;font-size:13px;line-height:1.5'>"
        "<div style='font-weight:700;margin-bottom:8px'>PARSED SLIDE</div>"
        "<div style='white-space:pre-wrap'>" + _html.escape(v["parse_text"]) + "</div></div>"
    )
    columns = "<div style='display:flex;gap:16px;align-items:flex-start'>" + element + parse + "</div>"
    blocks.append(
        "<div style='border-top:2px solid #bbb;padding:12px 0;margin-top:8px'>"
        + "<h2 style='margin:2px 0'>id " + str(v["id"]) + "</h2>" + meta + columns + "</div>"
    )

display(HTML("".join(blocks)))

## My scores


In [ ]:
my_verdicts = [
    {"id": 12,  "human": "covered"},
    {"id": 15,  "human": "missing"},
    {"id": 16,  "human": "covered"},
    {"id": 17,  "human": "missing"},
    {"id": 45,  "human": "covered"},
    {"id": 48,  "human": "covered"},
    {"id": 53,  "human": "covered"},
    {"id": 58,  "human": "covered"},
    {"id": 72,  "human": "covered"},
    {"id": 112, "human": "covered"},
    {"id": 115, "human": "covered"},
    {"id": 117, "human": "missing"},
    {"id": 121, "human": "missing"},
    {"id": 122, "human": "covered"},
    {"id": 128, "human": "covered"},
    {"id": 143, "human": "covered"},
    {"id": 219, "human": "covered"},
    {"id": 234, "human": "missing"},
    {"id": 262, "human": "covered"},  
    {"id": 283, "human": "covered"}, 
]

### Reasons

- **15**: Vier Linien werden genannt, aber die Kernaussage „unterschiedliche Steigung" wird widersproche. der Parse beschreibt Linie 1–3 als parallel zueinander. Außerdem verläuft die grüne Linie laut Parse nicht durch den Zwischenraum, sondern schneidet beide Klassen.
- **17**: Der Parse nennt drei korrekt trennende Linien, hebt aber keine einzelne als diejenige mit deutlichem Abstand zu beiden Klassen hervor. Der Margin-Gedanke (gute Trennung) fehlt komplett.
- **117**: Im Parse sind nur Kreise und Dreiecke beschrieben. Haken-/Häkchen-Markierungen an Punkten tauchen nirgends auf.
- **121**: Direkter Widerspruch. Der Parse sagt „Die beiden Punktwolken überlappen sich teilweise", das Golden-Element sagt, die Klassen seien getrennt.
- **234**: Die Annäherung an die Validierungskurve ist abgedeckt („Ab Lernzyklus 600 nähern sich Validierungs- und Testkurve einander an"), aber dem kleinere Minimum der Testkurve wird widersprochen. Laut Parse verläuft sie „etwas höher" als die Validierungskurve.
- **258**: Im Parse wird zwar die rote Entscheidungsgrenze beschrieben, aber eine ideale „Trennlinie“ wird an keiner Stelle erwähnt.

In [ ]:

rows = []
for m in my_verdicts:
    v = verdicts_by_id[m["id"]]
    rows.append({
        "id": m["id"],
        "slide": v["slide_id"],
        "typ": v["typ"],
        "llm": v["verdict"],
        "human": m["human"],
        "agree": v["verdict"] == m["human"],
    })

df = pd.DataFrame(rows)
agreement = df["agree"].mean() if len(df) else float("nan")
print(f"n={len(df)} | agreement={agreement * 100:.2f} %")

df.to_csv(VERDICTS_DIR / "parsing_judge_validation.csv", index=False, encoding="utf-8")
display(df)

## Agreement summary

In [ ]:
summary = (df.groupby("typ")["agree"].agg(["mean", "count"]).reset_index()
             .rename(columns={"mean": "agreement", "count": "n"}))
summary = pd.concat(
    [summary, pd.DataFrame([{"typ": "Gesamt", "agreement": df["agree"].mean(), "n": len(df)}])],
    ignore_index=True,
)
summary["agreement"] = (summary["agreement"] * 100).round(2).astype(str) + " %"

summary.to_csv(VERDICTS_DIR / "parsing_judge_validation_summary.csv", index=False, encoding="utf-8")
display(summary.style.hide(axis="index"))